# Look Inside Gemma 3 on a Free TPU
### Extracting activations with JAX and Keras

Language models can feel like black boxes, but you can open one up on a free TPU in about twenty minutes.

In this notebook we load **Gemma 3 1B** on a free TPU (Kaggle or Colab) and:

1. **Extract activations**: the model's hidden state after every one of its layers, with one compiled JAX function sharded across all TPU cores.
2. **Use the logit lens**: read what the model "would answer" at each layer.
3. **Find a language direction**: where Gemma separates Arabic from English, and where it realises that two sentences mean the same thing.
4. **Steer the model**: add that direction while it generates and watch the language of its answer change.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruqyai/ruqyai.github.io/blob/main/_notebooks/gemma3_tpu_activations.ipynb)

Author: Ruqiya Bin Safi ([ruqyai.github.io](https://ruqyai.github.io)). Companion article: [Look Inside Gemma 3 on a Free TPU](https://ruqyai.github.io/posts/2026/10/gemma3-tpu-activations/) · [بالعربية](https://ruqyai.github.io/ar/posts/2026/10/gemma3-tpu-activations/).

## 1. Setup

**On Kaggle**
1. Accept the Gemma license once on the [Gemma 3 model page](https://www.kaggle.com/models/keras/gemma3).
2. In the sidebar: **Settings → Accelerator → TPU**, and **Internet → On**.

**On Google Colab**
1. Accept the Gemma license once on [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) on Hugging Face.
2. Create a Hugging Face access token (Settings → Access Tokens, type *Read*), and add it in Colab's **🔑 Secrets** panel as `HF_TOKEN`, with notebook access on.
3. **Runtime → Change runtime type → TPU**.

Then run the cells in order. The notebook works with however many TPU cores you get.

In [ ]:
%pip install -q -U keras-hub arabic-reshaper python-bidi

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"   # Keras 3 runs on JAX, the TPU-native framework

import gc
import logging
import time
import psutil
import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
import keras
import keras_hub
import matplotlib
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display

# On TPU, Keras first tries the Splash flash-attention kernel, which needs sequence lengths that are
# multiples of 128. For our short inputs it logs an error and falls back to standard attention,
# which gives the same result. Hide those messages.
logging.getLogger("absl").setLevel(logging.CRITICAL)

print("Keras", keras.__version__, "| KerasHub", keras_hub.__version__, "| JAX", jax.__version__)
print("Devices:", jax.devices())
N_DEVICES = len(jax.devices())

def ram(tag=""):
    m = psutil.virtual_memory()
    print(f"RAM {tag}: {m.used / 1e9:.1f} GB used of {m.total / 1e9:.1f} GB")

ram("after imports")

In [ ]:
# Brand colours used in every figure, and a helper so Arabic renders correctly in matplotlib.
GREEN, MINT, HEAT, CREAM, INK, MUTED, PAPER = "#0D6B59", "#5FC8AE", "#F0A53A", "#EFE2B8", "#14211C", "#56655F", "#F3F6F4"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": "#D5DED9",
    "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
    "font.family": "DejaVu Sans",   # DejaVu Sans includes Arabic glyphs
})
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

NATIVE_RTL = tuple(int(x) for x in matplotlib.__version__.split(".")[:2]) >= (3, 11)

def ar(text):
    """Shape and reorder Arabic for matplotlib versions that can't do it themselves."""
    return text if NATIVE_RTL else get_display(arabic_reshaper.reshape(text))

def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=200, bbox_inches="tight")

## 2. Load Gemma 3 1B

We load the instruction-tuned 1B model in `bfloat16`, the number format TPUs are built around.

In [ ]:
MODEL = "1b"          # switch to "270m" if your runtime runs out of RAM

PRESETS = {"1b": ("gemma3_instruct_1b", "google/gemma-3-1b-it"),
           "270m": ("gemma3_instruct_270m", "google/gemma-3-270m-it")}
PRESET = PRESETS[MODEL][0]                        # Kaggle: Keras weights
try:
    from google.colab import userdata             # Colab: the same model from Hugging Face
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    PRESET = "hf://" + PRESETS[MODEL][1]
except ImportError:
    pass

causal_lm = keras_hub.models.Gemma3CausalLM.from_preset(PRESET, dtype="bfloat16")
backbone = causal_lm.backbone
tokenizer = causal_lm.preprocessor.tokenizer
gc.collect()
ram("after loading Gemma")

N_LAYERS = len(backbone.transformer_layers)
HIDDEN = backbone.hidden_dim
print(f"{N_LAYERS} layers, hidden size {HIDDEN}, vocabulary {backbone.vocabulary_size:,}")

In [ ]:
BOS = tokenizer.start_token_id
PAD = tokenizer.pad_token_id
END_OF_TURN = tokenizer.token_to_id("<end_of_turn>")

def chat(prompt):
    return f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

def encode(texts, seq_len):
    """Tokenise to a fixed length: same shape every call means JAX compiles only once."""
    ids = np.full((len(texts), seq_len), PAD, dtype="int32")
    mask = np.zeros((len(texts), seq_len), dtype="int32")
    for i, t in enumerate(texts):
        toks = [BOS] + [int(x) for x in np.asarray(tokenizer(t)).reshape(-1)]
        toks = toks[:seq_len]
        ids[i, :len(toks)] = toks
        mask[i, :len(toks)] = 1
    return ids, mask

def token_text(i):
    return tokenizer.id_to_token(int(i)).replace("▁", " ")

## 3. Extract activations

A Gemma backbone is a stack of decoder blocks. We rebuild its forward pass as a Keras functional model
that returns the hidden state after **every** block, sharing the original weights. Then we check that the
last block (after the final norm) matches the original backbone exactly.

In [ ]:
token_ids = keras.Input(shape=(None,), dtype="int32", name="token_ids")
padding_mask = keras.Input(shape=(None,), dtype="int32", name="padding_mask")

x = backbone.token_embedding(token_ids)
x = x * keras.ops.cast(keras.ops.sqrt(HIDDEN), x.dtype)     # Gemma scales embeddings by sqrt(hidden)
hidden_states = []
for block in backbone.transformer_layers:
    x = block(x, padding_mask=padding_mask)
    hidden_states.append(x)
extractor = keras.Model([token_ids, padding_mask], hidden_states)

ids, mask = encode(["Activations are the model's working memory."], 16)
reference = backbone({"token_ids": ids, "padding_mask": mask})
ours = backbone.layer_norm(extractor([ids, mask])[-1])
print("Matches the original backbone:",
      np.allclose(np.asarray(reference, "float32"), np.asarray(ours, "float32"), atol=1e-2))

### One compiled function, sharded across every TPU core

`stateless_call` turns the Keras model into a pure function of (weights, inputs), which is exactly what
`jax.jit` wants. A `Mesh` over all devices lets us **replicate** the weights and **split the batch**,
so each core processes its own slice of sentences in parallel.

Inside the compiled function we also pool the activations, so only small arrays travel back from the TPU:
- `last`: the hidden state at each sentence's final token (what the model is about to predict from)
- `mean`: the average over the sentence's tokens (skipping `<bos>`, whose activations are unusually large)

In [ ]:
mesh = Mesh(np.array(jax.devices()), ("batch",))
replicated = NamedSharding(mesh, P())
batch_sharded = NamedSharding(mesh, P("batch"))

weights = jax.device_put(
    ([v.value for v in extractor.trainable_variables],
     [v.value for v in extractor.non_trainable_variables]),
    replicated)

@jax.jit
def activations(weights, ids, mask):
    outs, _ = extractor.stateless_call(weights[0], weights[1], [ids, mask])
    h = jnp.stack(outs, axis=1).astype(jnp.float32)              # (batch, layers, seq, hidden)
    last_pos = mask.sum(axis=1) - 1
    last = h[jnp.arange(h.shape[0]), :, last_pos]                # (batch, layers, hidden)
    body = mask.at[:, 0].set(0)[:, None, :, None]                # every real token except <bos>
    mean = (h * body).sum(axis=2) / jnp.maximum(body.sum(axis=2), 1)
    return last, mean

def run_activations(texts, seq_len=64):
    """Pad the batch to a multiple of the device count, shard it, run, and unpad."""
    ids, mask = encode(texts, seq_len)
    n = len(texts)
    pad = (-n) % N_DEVICES
    if pad:
        ids = np.concatenate([ids, np.repeat(ids[:1], pad, 0)])
        mask = np.concatenate([mask, np.repeat(mask[:1], pad, 0)])
    last, mean = activations(weights, jax.device_put(ids, batch_sharded), jax.device_put(mask, batch_sharded))
    return np.asarray(last)[:n], np.asarray(mean)[:n]

last, mean = run_activations(["Hello TPU!", "مرحباً يا TPU!"])
print("last:", last.shape, "mean:", mean.shape)

### How fast is it, and what slows it down?

Three measurements on the same 64 sentences:
1. **First call**: includes XLA compilation, which happens once per input shape.
2. **Later calls**: the compiled program, sharded across all cores.
3. **One core only**: the same program without sharding.

And a warning in numbers: if every batch has a different length, JAX recompiles every time.

In [ ]:
bench = [f"Sentence number {i} about tensors, chips and attention." for i in range(64)]

def timed(fn, repeat=5):
    out = fn()
    jax.block_until_ready(out)
    t = time.perf_counter()
    for _ in range(repeat):
        jax.block_until_ready(fn())
    return (time.perf_counter() - t) / repeat

ids_b, mask_b = encode(bench, 64)
sharded_args = (jax.device_put(ids_b, batch_sharded), jax.device_put(mask_b, batch_sharded))

activations_fresh = jax.jit(activations.__wrapped__)   # a fresh cache, so we can time compilation
t = time.perf_counter()
jax.block_until_ready(activations_fresh(weights, *sharded_args))
t_first = time.perf_counter() - t
t_sharded = timed(lambda: activations_fresh(weights, *sharded_args))

one = jax.devices()[0]
weights_one = jax.device_put(weights, one)
single_args = (jax.device_put(ids_b, one), jax.device_put(mask_b, one))
t_single = timed(lambda: activations_fresh(weights_one, *single_args))

lengths = [40, 48, 56, 64, 72]
t = time.perf_counter()
for L in lengths:                                       # a new shape each time: a new compilation each time
    i_, m_ = encode(bench, L)
    jax.block_until_ready(activations_fresh(weights, jax.device_put(i_, batch_sharded), jax.device_put(m_, batch_sharded)))
t_varying = (time.perf_counter() - t) / len(lengths)

compiled_label = f"Compiled,\nall {N_DEVICES} cores" if N_DEVICES > 1 else "Same shape again\n(already compiled)"
timings = {"First call\n(compile + run)": t_first, compiled_label: t_sharded}
if N_DEVICES > 1:
    timings["Compiled,\n1 core"] = t_single
timings["New length\nevery batch"] = t_varying
for k, v in timings.items():
    print(f"{k.replace(chr(10), ' '):32s} {v * 1000:9.1f} ms")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
bars = ax.bar(list(timings), [v * 1000 for v in timings.values()], color=[MUTED, GREEN, MINT, HEAT][: len(timings) - 1] + [HEAT], width=0.6)
ax.bar_label(bars, labels=[f"{v * 1000:,.0f} ms" for v in timings.values()], padding=3, color=INK)
ax.set_ylabel("Time per batch of 64 sentences (ms)")
ax.set_yscale("log")
ax.set_title(f"Extracting all {N_LAYERS} layers on {jax.devices()[0].device_kind}", loc="left", color=INK)
save(fig, "01-timing")
plt.show()

## 4. The logit lens

The final layer turns a hidden state into a prediction: final norm, then multiply by the embedding matrix.
The **logit lens** applies that same read-out to every intermediate layer, so we can watch the answer form.

In [ ]:
SOFT_CAP = getattr(backbone, "final_logit_soft_cap", None)

@jax.jit
def lens(h):                                         # h: (layers, hidden), float32
    logits = backbone.token_embedding(backbone.layer_norm(h.astype(jnp.bfloat16)), reverse=True)
    logits = logits.astype(jnp.float32)
    if SOFT_CAP:
        logits = jnp.tanh(logits / SOFT_CAP) * SOFT_CAP
    return jax.nn.softmax(logits, axis=-1)

def logit_lens(prompt, answer):
    last, _ = run_activations([prompt])
    probs = np.asarray(lens(jnp.asarray(last[0])))       # (layers, vocab)
    answer_id = int(np.asarray(tokenizer(answer)).reshape(-1)[0])
    top = probs.argmax(-1)
    return probs[:, answer_id], [token_text(t) for t in top], token_text(answer_id)

prompts = [
    ("The capital of Saudi Arabia is", " Riyadh", "English"),
    ("عاصمة المملكة العربية السعودية هي", " الرياض", "Arabic"),
]
lens_results = {}
fig, ax = plt.subplots(figsize=(8, 3.8))
for (prompt, answer, label), colour in zip(prompts, [HEAT, GREEN]):
    p_answer, tops, answer_tok = logit_lens(prompt, answer)
    lens_results[label] = (p_answer, tops)
    ax.plot(range(1, N_LAYERS + 1), p_answer, marker="o", ms=4, lw=2.2, color=colour,
            label=f"{label}: P({ar(answer_tok.strip())})")
    print(f"\n{label} | top token per layer:")
    print(" | ".join(f"{i + 1}:{t.strip()}" for i, t in enumerate(tops)))
ax.set_xlabel("Layer")
ax.set_ylabel("Probability of the right answer")
ax.set_title("When does Gemma 3 'know' the answer?", loc="left")
ax.legend(frameon=False)
save(fig, "02-logit-lens")
plt.show()

## 5. A direction for language

Forty short sentences, each in Arabic and in English. For every layer we ask two questions:

- **Can we tell the languages apart?** Learn the Arabic−English difference of means on half the pairs and test it on the other half.
- **Does the model know two sentences mean the same thing?** Remove each language's average, then check whether each Arabic sentence is closest to *its own* English translation.

In [ ]:
PAIRS = [
    ("The sun rises in the east.", "تشرق الشمس من الشرق."),
    ("I drink coffee every morning.", "أشرب القهوة كل صباح."),
    ("The library opens at nine.", "تفتح المكتبة في الساعة التاسعة."),
    ("My brother plays football on Fridays.", "يلعب أخي كرة القدم أيام الجمعة."),
    ("The train is late today.", "القطار متأخر اليوم."),
    ("She is reading a long novel.", "إنها تقرأ رواية طويلة."),
    ("The weather is very hot in summer.", "الطقس حار جداً في الصيف."),
    ("We planted three trees in the garden.", "زرعنا ثلاث أشجار في الحديقة."),
    ("The children are sleeping now.", "الأطفال نائمون الآن."),
    ("This road leads to the sea.", "هذا الطريق يؤدي إلى البحر."),
    ("He forgot his keys at home.", "نسي مفاتيحه في البيت."),
    ("The market is crowded in the evening.", "السوق مزدحم في المساء."),
    ("I am learning to cook rice.", "أتعلم طبخ الأرز."),
    ("The cat is sitting on the chair.", "القطة تجلس على الكرسي."),
    ("Water boils at one hundred degrees.", "يغلي الماء عند مئة درجة."),
    ("They travelled to Jeddah by car.", "سافروا إلى جدة بالسيارة."),
    ("The meeting starts after lunch.", "يبدأ الاجتماع بعد الغداء."),
    ("Please close the window.", "من فضلك أغلق النافذة."),
    ("The moon is bright tonight.", "القمر ساطع الليلة."),
    ("My phone battery is empty.", "بطارية هاتفي فارغة."),
    ("The doctor told him to rest.", "طلب منه الطبيب أن يرتاح."),
    ("We need more chairs for the guests.", "نحتاج إلى كراسٍ أكثر للضيوف."),
    ("The river flows into the sea.", "يصب النهر في البحر."),
    ("I wrote a letter to my friend.", "كتبت رسالة إلى صديقي."),
    ("The bread is still warm.", "الخبز ما زال دافئاً."),
    ("Birds fly south in winter.", "تهاجر الطيور جنوباً في الشتاء."),
    ("The computer is too slow.", "الحاسوب بطيء جداً."),
    ("She paints pictures of the desert.", "إنها ترسم لوحات للصحراء."),
    ("The shop closes at midnight.", "يغلق المتجر عند منتصف الليل."),
    ("He runs five kilometres every day.", "يركض خمسة كيلومترات كل يوم."),
    ("The exam was easier than I expected.", "كان الاختبار أسهل مما توقعت."),
    ("Our team won the match.", "فاز فريقنا في المباراة."),
    ("The milk is in the fridge.", "الحليب في الثلاجة."),
    ("I lost my umbrella in the rain.", "أضعت مظلتي تحت المطر."),
    ("The museum has many old coins.", "في المتحف عملات قديمة كثيرة."),
    ("My grandmother tells wonderful stories.", "جدتي تحكي قصصاً رائعة."),
    ("The plane landed safely.", "هبطت الطائرة بسلام."),
    ("We watched the stars from the roof.", "شاهدنا النجوم من السطح."),
    ("This tea needs more sugar.", "هذا الشاي يحتاج إلى سكر أكثر."),
    ("The new bridge is very long.", "الجسر الجديد طويل جداً."),
]
english = [e for e, _ in PAIRS]
arabic = [a for _, a in PAIRS]

_, mean_en = run_activations(english)       # (40, layers, hidden)
_, mean_ar = run_activations(arabic)

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(PAIRS))
train, test = order[: len(PAIRS) // 2], order[len(PAIRS) // 2:]

separation, alignment = [], []
for layer in range(N_LAYERS):
    en, a = mean_en[:, layer], mean_ar[:, layer]

    # 1) language direction: learned on train pairs, tested on held-out pairs
    direction = a[train].mean(0) - en[train].mean(0)
    threshold = ((a[train] @ direction).mean() + (en[train] @ direction).mean()) / 2
    correct = np.concatenate([(a[test] @ direction) > threshold, (en[test] @ direction) <= threshold])
    separation.append(correct.mean())

    # 2) meaning: remove each language's average, then match each Arabic sentence to its translation
    en_c = en - en.mean(0)
    a_c = a - a.mean(0)
    en_c /= np.linalg.norm(en_c, axis=1, keepdims=True)
    a_c /= np.linalg.norm(a_c, axis=1, keepdims=True)
    similarity = a_c @ en_c.T
    alignment.append((similarity.argmax(1) == np.arange(len(PAIRS))).mean())

fig, ax = plt.subplots(figsize=(8, 3.8))
layers = range(1, N_LAYERS + 1)
ax.plot(layers, separation, lw=2.4, color=HEAT, marker="o", ms=4, label="Tells Arabic from English (held-out accuracy)")
ax.plot(layers, alignment, lw=2.4, color=GREEN, marker="o", ms=4, label="Finds the matching translation (out of 40)")
ax.axhline(1 / len(PAIRS), color=MUTED, ls=":", lw=1)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Layer")
ax.set_ylabel("Accuracy")
ax.set_title("Language versus meaning, layer by layer", loc="left")
ax.legend(frameon=False, loc="lower right")
save(fig, "03-language-vs-meaning")
plt.show()
best = int(np.argmax(alignment))
print(f"Best translation matching at layer {best + 1}: {alignment[best]:.0%}")

In [ ]:
# A 2-D picture of four layers: each dot is one sentence, projected on its layer's top two principal components.
show = sorted({1, N_LAYERS // 3, best + 1, N_LAYERS})
fig, axes = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3.2))
for axis, layer in zip(axes, show):
    both = np.concatenate([mean_en[:, layer - 1], mean_ar[:, layer - 1]])
    both = both - both.mean(0)
    _, _, vt = np.linalg.svd(both, full_matrices=False)
    xy = both @ vt[:2].T
    n = len(PAIRS)
    for i in range(n):
        axis.plot([xy[i, 0], xy[n + i, 0]], [xy[i, 1], xy[n + i, 1]], color="#D5DED9", lw=0.6, zorder=1)
    axis.scatter(*xy[:n].T, s=18, color=HEAT, label="English", zorder=2)
    axis.scatter(*xy[n:].T, s=18, color=GREEN, label=ar("عربي"), zorder=2)
    axis.set_title(f"Layer {layer}", color=INK)
    axis.set_xticks([]); axis.set_yticks([])
axes[0].legend(frameon=False, loc="best")
fig.suptitle("Each line joins a sentence to its translation", x=0.01, y=1.06, ha="left", color=INK)
save(fig, "04-pca-layers")
plt.show()

## 6. Steering: push an English prompt towards Arabic

We now rebuild the model once more, this time with a third input: a **steering vector** added to the hidden state
after one chosen layer. Each row in the batch gets its own vector, so one compiled call tries several strengths
at once, one per TPU core.

The vector is the Arabic−English difference of means at that layer, multiplied by a strength `c`.
`c = 0` is the untouched model.

In [ ]:
STEER_LAYER = best              # the layer where meaning is most shared across languages
steer_input = keras.Input(shape=(HIDDEN,), dtype="float32", name="steer")
t_in = keras.Input(shape=(None,), dtype="int32", name="token_ids")
m_in = keras.Input(shape=(None,), dtype="int32", name="padding_mask")

y = backbone.token_embedding(t_in)
y = y * keras.ops.cast(keras.ops.sqrt(HIDDEN), y.dtype)
for i, block in enumerate(backbone.transformer_layers):
    y = block(y, padding_mask=m_in)
    if i == STEER_LAYER:
        y = y + keras.ops.cast(keras.ops.expand_dims(steer_input, 1), y.dtype)
y = backbone.layer_norm(y)
steered = keras.Model([t_in, m_in, steer_input], backbone.token_embedding(y, reverse=True))

steer_weights = jax.device_put(
    ([v.value for v in steered.trainable_variables],
     [v.value for v in steered.non_trainable_variables]),
    replicated)

@jax.jit
def next_token(weights, ids, mask, steer, pos):
    logits, _ = steered.stateless_call(weights[0], weights[1], [ids, mask, steer])
    return logits[jnp.arange(ids.shape[0]), pos].argmax(-1)

def generate_steered(prompt, strengths, max_new=40, seq_len=96):
    """Greedy decoding at a fixed length, so the whole loop reuses one compiled program."""
    rows = len(strengths) + (-len(strengths)) % N_DEVICES
    ids, mask = encode([chat(prompt)] * rows, seq_len)
    direction = mean_ar[:, STEER_LAYER].mean(0) - mean_en[:, STEER_LAYER].mean(0)
    c = np.array(list(strengths) + [0.0] * (rows - len(strengths)), "float32")
    steer = jax.device_put(c[:, None] * direction[None, :], batch_sharded)
    pos = mask.sum(1) - 1
    done = np.zeros(rows, bool)
    for _ in range(max_new):
        if pos.max() + 1 >= seq_len:
            break
        tok = np.asarray(next_token(steer_weights, jax.device_put(ids, batch_sharded),
                                    jax.device_put(mask, batch_sharded), steer, jnp.asarray(pos)))
        for r in range(rows):
            if not done[r]:
                pos[r] += 1
                ids[r, pos[r]] = tok[r]
                mask[r, pos[r]] = 1
                done[r] = tok[r] == END_OF_TURN
        if done.all():
            break
    start = len(encode([chat(prompt)], seq_len)[1][0].nonzero()[0])
    return [tokenizer.detokenize(ids[r, start: pos[r] + 1]) for r in range(len(strengths))]

STRENGTHS = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]
outputs = generate_steered("Write one short sentence about the ocean.", STRENGTHS)
for c, text in zip(STRENGTHS, outputs):
    print(f"c = {c:>3}:  {str(text).replace('<end_of_turn>', '').strip()}")

## 7. What we did

- Rebuilt Gemma 3's forward pass to expose all 26 hidden states, and verified it against the original.
- Compiled one JAX function, replicated the weights and sharded the batch across every TPU core.
- Kept every input the same shape so XLA compiles once, and measured what happens when we don't.
- Watched answers form layer by layer with the logit lens.
- Found where Gemma separates Arabic from English, and where it aligns their meaning.
- Steered generation with a single vector.

**Try next:** swap the sentence pairs for another concept (formal vs informal, positive vs negative), try the
`gemma3_instruct_4b_text` preset, or read how the same idea explains refusal in
[Exploring the Refusal Direction in Gemma 3 on TPU](https://ruqyai.github.io/posts/2026/04/refusal-direction-gemma-3-tpu/).

The figures are saved in `figures/`.

In [ ]:
# Save every number from this run next to the figures, then zip them into one file to download.
import shutil
with open(f"{FIG_DIR}/results.txt", "w", encoding="utf-8") as f:
    f.write(f"Device: {jax.devices()[0].device_kind} x {N_DEVICES}\n")
    f.write(f"Keras {keras.__version__} | KerasHub {keras_hub.__version__} | JAX {jax.__version__}\n\n")
    f.write("== Timing (ms per batch of 64) ==\n")
    for k, v in timings.items():
        f.write(f"{k.replace(chr(10), ' ')}: {v * 1000:.1f}\n")
    f.write("\n== Logit lens ==\n")
    for label, (p_answer, tops) in lens_results.items():
        f.write(f"{label}\n")
        for i, (p, t) in enumerate(zip(p_answer, tops)):
            f.write(f"  layer {i + 1:2d}: P(answer)={p:.4f}  top={t.strip()}\n")
    f.write("\n== Language vs meaning ==\n")
    for i, (s_, a_) in enumerate(zip(separation, alignment)):
        f.write(f"  layer {i + 1:2d}: separation={s_:.3f}  translation_match={a_:.3f}\n")
    f.write(f"\n== Steering at layer {STEER_LAYER + 1} ==\n")
    for c, text in zip(STRENGTHS, outputs):
        f.write(f"c = {c}: {str(text).replace('<end_of_turn>', '').strip()}\n")
shutil.make_archive("gemma3-tpu-results", "zip", FIG_DIR)
print("Download gemma3-tpu-results.zip from the Output panel (right sidebar, /kaggle/working).")